In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Dataset
from torch import manual_seed, nn, no_grad, optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

torch.manual_seed(10101010) # because we randomly sort the data inside the data loader

In [2]:
class TensorData(Dataset):
    def __init__(self, input_tensor, label_tensor):
        self.input = input_tensor
        self.labels = label_tensor

    def __len__(self):
        return self.input.size()[0]

    def __getitem__(self, index):
        return self.input[index], self.labels[index]

def load_data_pd():
    fence5 = "kaggle_train_5_fences.csv"
    fence7 = "kaggle_train_7_fences.csv"
    fence9 = "kaggle_train_9_fences.csv"
    bench = "kaggle_hidden_test_fences.csv"
    return {"fence5": pd.read_csv(fence5), "fence7": pd.read_csv(fence7), "fence9": pd.read_csv(fence9), "bench": pd.read_csv(bench)}

In [3]:
#Load Data numpy

data_pd = load_data_pd()
data_pd_5 = data_pd["fence5"].copy()
data_pd_7 = data_pd["fence7"].copy()
data_pd_9 = data_pd["fence9"].copy()
data_pd_bench = data_pd["bench"].copy()

data_pd_bench = data_pd_bench.fillna(0)


#Padding fence5 and fence7 with zeros

data_pd_5.insert(6, '8', 0)
data_pd_5.insert(6, '7', 0)
data_pd_5.insert(6, '6', 0)
data_pd_5.insert(6, '5', 0)

data_pd_7.insert(8, '8', 0)
data_pd_7.insert(8, '7', 0)

#Isolating only the inputs

data_pd_5_train = data_pd_5.loc[:, ['0', '1', '2', '3', '4', '5', '6', '7', '8']]
data_pd_5_train.columns = pd.to_numeric(data_pd_5_train.columns)

data_pd_7_train = data_pd_7.loc[:, ['0', '1', '2', '3', '4', '5', '6', '7', '8']]
data_pd_7_train.columns = pd.to_numeric(data_pd_7_train.columns)

data_pd_9_train = data_pd_9.loc[:, ['0', '1', '2', '3', '4', '5', '6', '7', '8']]
data_pd_9_train.columns = pd.to_numeric(data_pd_9_train.columns)

data_pd_bench = data_pd_bench.loc[:, ['0', '1', '2', '3', '4', '5', '6', '7', '8']]
data_pd_bench.columns = pd.to_numeric(data_pd_bench.columns)

#Sort function (smallest to biggest) for the fence lengths

def sort_fences(data):
    data_trans = data.copy()
    data_trans = data_trans.T
    for col in data_trans:
        data_trans[col] = data_trans[col].sort_values(ignore_index=True)
    return data_trans.T

#Sort data
data_pd_5_train_sorted = sort_fences(data_pd_5_train)

data_pd_7_train_sorted = sort_fences(data_pd_7_train)

data_pd_9_train_sorted = sort_fences(data_pd_9_train)

data_pd_bench_sorted = sort_fences(data_pd_bench)

#Full dataframes with sorted values
data_pd_5_sorted = data_pd_5.copy()
data_pd_5_sorted[['0', '1', '2', '3', '4', '5', '6', '7', '8']] = data_pd_5_train_sorted[[0, 1, 2, 3, 4, 5, 6, 7, 8]]
#data_pd_5_sorted.rename(columns={5: '5', 6: '6', 7: '7', 8: '8'})

data_pd_7_sorted = data_pd_7.copy()
data_pd_7_sorted[['0', '1', '2', '3', '4', '5', '6', '7', '8']] = data_pd_7_train_sorted[[0, 1, 2, 3, 4, 5, 6, 7, 8]]
#data_pd_7_sorted.rename(columns={7: '7', 8: '8'})

data_pd_9_sorted = data_pd_9.copy()
data_pd_9_sorted[['0', '1', '2', '3', '4', '5', '6', '7', '8']] = data_pd_9_train_sorted[[0, 1, 2, 3, 4, 5, 6, 7, 8]]

data_pd_combined = pd.concat([data_pd_5_sorted, data_pd_7_sorted, data_pd_9_sorted])

In [4]:
print(data_pd_bench_sorted)

              0         1         2         3          4          5  \
0      0.000000  0.000000  0.000000  0.000000   1.262297   7.692489   
1      0.000000  0.000000  0.000000  0.000000   0.351311   2.810874   
2      0.000000  0.000000  0.000000  0.000000   0.704452   3.082461   
3      0.000000  0.000000  0.000000  0.000000   0.126651   1.286372   
4      0.000000  0.000000  0.000000  0.000000  18.842998  23.444140   
...         ...       ...       ...       ...        ...        ...   
44995  1.610204  2.031407  2.353356  5.336264   6.997141   9.063055   
44996  0.043520  0.179046  0.273191  0.839162   3.080773   4.675428   
44997  2.266608  4.456923  5.011706  8.246457  12.640853  14.783877   
44998  0.188096  0.252918  0.257124  0.273025   0.656424   0.864561   
44999  1.206239  2.206838  3.535355  4.539612   4.608005   5.208765   

               6           7           8  
0      20.267731   26.539881   27.156419  
1       3.830812    4.451284    7.171948  
2      57.520394  

In [5]:
#Homogenize the data

#Original Inputs 
f0 = data_pd_combined['0']
f1 = data_pd_combined['1']
f2 = data_pd_combined['2']
f3 = data_pd_combined['3']
f4 = data_pd_combined['4']
f5 = data_pd_combined['5']
f6 = data_pd_combined['6']
f7 = data_pd_combined['7']
f8 = data_pd_combined['8']

#Original Inputs Bench Data
f0b = data_pd_bench_sorted[0]
f1b = data_pd_bench_sorted[1]
f2b = data_pd_bench_sorted[2]
f3b = data_pd_bench_sorted[3]
f4b = data_pd_bench_sorted[4]
f5b = data_pd_bench_sorted[5]
f6b = data_pd_bench_sorted[6]
f7b = data_pd_bench_sorted[7]
f8b = data_pd_bench_sorted[8]

#Homogeneous inputs (multiply biggest (f8) by everything else; multiply guaranteed non-zeros (f8-f4 incl.))
x0 = f0*f8 #? should work maybe
x1 = f1*f8
x2 = f2*f8
x3 = f3*f8
x4 = f4*f8
x5 = f5*f8
x6 = f6*f8
x7 = f7*f8
x8 = f8*f8

#Homogeneous inputs BENCH
x0b = f0b*f8b #? should work maybe
x1b = f1b*f8b
x2b = f2b*f8b
x3b = f3b*f8b
x4b = f4b*f8b
x5b = f5b*f8b
x6b = f6b*f8b
x7b = f7b*f8b
x8b = f8b*f8b

#Homo data
data_nh = data_pd_combined.copy()
data_nh['0'] = x0
data_nh['1'] = x1
data_nh['2'] = x2
data_nh['3'] = x3
data_nh['4'] = x4
data_nh['5'] = x5
data_nh['6'] = x6
data_nh['7'] = x7
data_nh['8'] = x8

#Homo data Bench
data_nh_bench = data_pd_bench_sorted.copy()
data_nh_bench['0'] = x0b
data_nh_bench['1'] = x1b
data_nh_bench['2'] = x2b
data_nh_bench['3'] = x3b
data_nh_bench['4'] = x4b
data_nh_bench['5'] = x5b
data_nh_bench['6'] = x6b
data_nh_bench['7'] = x7b
data_nh_bench['8'] = x8b


data_nd = data_pd_combined.copy()
data_nd['0'] = x0
data_nd['1'] = x1
data_nd['2'] = x2
data_nd['3'] = x3
data_nd['4'] = x4
data_nd['5'] = x5
data_nd['6'] = x6
data_nd['7'] = x7
data_nd['8'] = x8


#Homo data scaled -> only inputs
data_nh_scaled = data_nh.loc[:, ['0', '1', '2', '3', '4', '5', '6', '7', '8']]
data_nh_scaled = data_nh_scaled.div(data_nh['area'], axis=0)

#Dimensionless data -> inputs + area (divided by x8, so new inputs are x0 to x7)
data_nd = data_nh.loc[:, ['0', '1', '2', '3', '4', '5', '6', '7', '8', 'area']]
data_nd = data_nd.div(data_nh['8'], axis=0)
data_nd['CE'] = data_nh['CE']

#Dimensionless data -> inputs + area (divided by x8, so new inputs are x0 to x7)
data_nd_bench = data_nh_bench.copy()
data_nd_bench = data_nd_bench.div(data_nd_bench['8'], axis=0)

print(data_nd_bench)

              0         1         2         3         4         5         6  \
0      0.000000  0.000000  0.000000  0.000000  0.001712  0.010431  0.027483   
1      0.000000  0.000000  0.000000  0.000000  0.006830  0.054647  0.074476   
2      0.000000  0.000000  0.000000  0.000000  0.000021  0.000093  0.001726   
3      0.000000  0.000000  0.000000  0.000000  0.004138  0.042029  0.064537   
4      0.000000  0.000000  0.000000  0.000000  0.001302  0.001621  0.003441   
...         ...       ...       ...       ...       ...       ...       ...   
44995  0.002213  0.002792  0.003235  0.007334  0.009617  0.012457  0.014576   
44996  0.000167  0.000687  0.001048  0.003220  0.011823  0.017943  0.020232   
44997  0.000652  0.001282  0.001442  0.002372  0.003636  0.004252  0.004660   
44998  0.025546  0.034349  0.034920  0.037080  0.089150  0.117417  0.126299   
44999  0.001153  0.002109  0.003379  0.004338  0.004404  0.004978  0.009998   

              7         8         0         1      

now lets split the data in to a training set and a validation set.

In [6]:
data_nd = data_nd.reset_index(drop=True)
data_nd_train = data_nd.sample(frac = 0.8 , random_state=1111,ignore_index=True)
data_nd_test = data_nd.drop(data_nd_train.index).reset_index(drop=True)

data_nd_x = data_nd[['0','1','2','3','4','5','6','7']]
data_nd_y = data_nd['area']
data_nd_L = data_nd['CE']

tensor_nd_x = torch.from_numpy(data_nd_x.to_numpy().copy()).type(torch.float32) #x
tensor_nd_y = torch.from_numpy(data_nd_y.to_numpy().copy()).type(torch.float32) #y
tensor_nd_L = torch.from_numpy(data_nd_L.to_numpy().copy()).type(torch.float32) #label

tensor_nd_y = tensor_nd_y.reshape(-1,1)
tensor_nd_L = tensor_nd_L.reshape(-1,1)

tensor_nd_full = {"x": tensor_nd_x, "y": tensor_nd_y, "L": tensor_nd_L}

#Bench data to tensor
data_nd_bench_x = data_nd_bench[['0','1','2','3','4','5','6','7']]
tensor_nd_bench = torch.from_numpy(data_nd_bench_x.to_numpy().copy()).type(torch.float32)
tensor_nd_bench = {"x": tensor_nd_bench}

data_nh = data_nh.reset_index(drop=True)
data_nh_train = data_nh.sample(frac = 0.8 , random_state=1111,ignore_index=True)
data_nh_test = data_nh.drop(data_nd_train.index).reset_index(drop=True)

data_nh_scaled = data_nh_scaled.reset_index(drop=True)
data_nh_scaled_train = data_nh_scaled.sample(frac = 0.8 , random_state=1111,ignore_index=True)
data_nh_scaled_test = data_nh_scaled.drop(data_nd_train.index).reset_index(drop=True)

#turning the training data to torch tensors

data_nd_train_x = data_nd_train[['0','1','2','3','4','5','6','7']]
data_nd_train_y = data_nd_train['area']
data_nd_train_L = data_nd_train['CE']

data_nd_test_x = data_nd_test[['0','1','2','3','4','5','6','7']]
data_nd_test_y = data_nd_test['area']
data_nd_test_L = data_nd_test['CE']

def remove_scale(data_with_scale):
    # Divide every fence length by longest fence for [-]
    # Must be done for test data also, so argmax is used

    values = data_with_scale.copy()
    for idx, row in enumerate(values):
        max_index = row.argmax()
        values[idx] = row / row[max_index]
        
    return values



tensor_nd_train_x = torch.from_numpy(data_nd_train_x.to_numpy().copy()).type(torch.float32) #x
tensor_nd_train_y = torch.from_numpy(data_nd_train_y.to_numpy().copy()).type(torch.float32) #y
tensor_nd_train_L = torch.from_numpy(data_nd_train_L.to_numpy().copy()).type(torch.float32) #label

tensor_nd_test_x = torch.from_numpy(data_nd_test_x.to_numpy().copy()).type(torch.float32) 
tensor_nd_test_y = torch.from_numpy(data_nd_test_y.to_numpy().copy()).type(torch.float32)
tensor_nd_test_L = torch.from_numpy(data_nd_test_L.to_numpy().copy()).type(torch.float32)


tensor_nd_train_y = tensor_nd_train_y.reshape(-1,1)
tensor_nd_train_L = tensor_nd_train_L.reshape(-1,1) 
tensor_nd_test_L = tensor_nd_test_L.reshape(-1,1) 
tensor_nd_test_y = tensor_nd_test_y.reshape(-1,1)

tensor_nd_train = {"x": tensor_nd_train_x, "y": tensor_nd_train_y, "L": tensor_nd_train_L}
tensor_nd_test = {"x": tensor_nd_test_x, "y": tensor_nd_test_y, "L": tensor_nd_test_L}


#tensor_nh_train = torch.from_numpy(data_nh_train.to_numpy() )
#tensor_nh_test = torch.from_numpy(data_nh_test.to_numpy() )

#tensor_nh_scaled_train = torch.from_numpy(data_nh_scaled_train.to_numpy()) 
#tensor_nh_scaled_test = torch.from_numpy(data_nh_scaled_test.to_numpy())

print(data_nd_x)
print(data_nd_bench_x)

              0         1         2         3         4         5         6  \
0      0.000000  0.000000  0.000000  0.000000  0.035524  0.270916  0.397326   
1      0.000000  0.000000  0.000000  0.000000  0.134408  0.138259  0.231789   
2      0.000000  0.000000  0.000000  0.000000  0.074568  0.432052  0.455399   
3      0.000000  0.000000  0.000000  0.000000  0.183678  0.421353  0.456577   
4      0.000000  0.000000  0.000000  0.000000  0.108067  0.113459  0.157686   
...         ...       ...       ...       ...       ...       ...       ...   
14995  0.026643  0.031158  0.036650  0.101088  0.106491  0.312210  0.348696   
14996  0.001981  0.039692  0.103625  0.310871  0.374972  0.454194  0.709128   
14997  0.041727  0.044596  0.112855  0.164175  0.183994  0.359263  0.685096   
14998  0.034749  0.082387  0.159107  0.185606  0.256508  0.268354  0.311765   
14999  0.026364  0.028840  0.068045  0.108452  0.152883  0.279111  0.289821   

              7  
0      0.532108  
1      0.570214

define what model we are gonna use

In [7]:
def binary_acuracy(predictions,labels_data): 
    return torch.sum( torch.round(predictions) == labels_data )/labels_data.shape[0]

In [8]:
result = binary_acuracy(torch.tensor((2.2,0.2,0.8,0.9)),torch.tensor((1.,0.,1.,0.)))
print(result)

print()

tensor(0.5000)



In [9]:
def make_model_IPFCE(width):  #Is polygonal fence Center-Enclosing
    return nn.Sequential(
        nn.Linear(8, width, bias=True),
        nn.ReLU(),
        nn.Linear(width, width, bias=True),
        nn.ReLU(),
        nn.Linear(width, width, bias=True),
        nn.ReLU(),
        nn.Linear(width, width, bias=True),
        nn.ReLU(),
        nn.Linear(width, 1, bias=True),
        nn.Sigmoid(),
    )# with this model we are going for a dimension less one 

In [10]:
def make_model_MAEBAPF(width = 8):  #Maximum area enclosed by a polygonal fence
    innput = 8
    output= 1
    return nn.Sequential(
        nn.Linear(innput, width, bias=True),
        nn.ReLU(),
        nn.Linear(width, width, bias=True),
        nn.ReLU(),
        nn.Linear(width, output, bias=True),
    ) # with this model we are going for a dimension less one 

In [11]:
def train_model(
    train_data:dict,
    test_data:dict,
    test_labels:bool,
    model,
    loss_fn,
    epochs:int,
    lr:float,
    batch_size:int,
    print_every:int,
    accuracy_fn=None,
):
    loss_dict = {"train": [], "test": [], "test_acc": []}

    # Initialize optimizer
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.1,patience=10)
    data_length = train_data['x'].shape[0]
    nr_batches = int(data_length/batch_size)
    #split in batches
    X=[]
    Y=[]
    L=[]
    
    for i in range(nr_batches): # only do full batches
        X.append(train_data['x'][i*batch_size:i*batch_size+batch_size,:])
        Y.append(train_data['y'][i*batch_size:i*batch_size+batch_size,:])
        L.append(train_data['L'][i*batch_size:i*batch_size+batch_size,:])
        
    
    # Print header.
    
    print(f"Epoch    Train loss      Test loss       Test accuracy")
    for epoch in range(epochs):
        epoch_loss_sum = 0
        models = []
        for i in range(nr_batches):
            x_batch = X[i]
            y_batch = Y[i]
            L_batch = L[i]
            
            # Reset optimizer gradients.
            optimizer.zero_grad()
               
            # Predict the output
            y_pred = model(x_batch)

            # Compute the loss
            if test_labels:
                loss = loss_fn(y_pred, L_batch)
            else :
                loss = loss_fn(y_pred, y_batch)
            epoch_loss_sum += loss.item()

            # Compute gradients according to newly computed loss.
            loss.backward()

            # Update the model parameters.
            optimizer.step()

        loss_dict["train"].append(epoch_loss_sum / nr_batches)

        with no_grad():
            test_pred = model(test_data['x'])
            if test_labels:
                test_loss = loss_fn(test_pred, test_data['L']) #if we are training for the fence encloseing we do this
                test_accuracy = accuracy_fn(test_pred , test_data['L'] )
                loss_dict["test_acc"].append(test_accuracy.item())
            else:
                test_loss = loss_fn(test_pred, test_data['y']) #if we are training for the maximum area
                loss_dict["test_acc"].append(0.0)
            loss_dict["test"].append(test_loss.item())
                

        if (epoch + 1) % print_every == 0:
            
            print(
                f"{epoch+1: <7}  {loss_dict['train'][-1]: <14.6e}  {loss_dict['test'][-1]: <13.6e}  {loss_dict['test_acc'][-1]: .6}"
            )
            models.append(model)
            

    return model, loss_dict

In [15]:
epochs = 500
print_every = 10
width = 40
lr = 5e-4
batch_size = 75


model = make_model_IPFCE(width)
#model = torch.compile(model)
loss_fn = nn.BCELoss()
accuracy_fn = binary_acuracy
labels = True
models__IPFCE, loss_dict__IPFCE = train_model(tensor_nd_train,tensor_nd_test,labels,model,loss_fn,epochs,lr,batch_size,print_every,accuracy_fn)



Epoch    Train loss      Test loss       Test accuracy
10       3.852711e-02    2.541117e-02    0.991667
20       2.923025e-02    2.025106e-02    0.992333
30       2.681479e-02    1.713065e-02    0.993
40       2.509577e-02    1.518296e-02    0.994333
50       2.381131e-02    1.442832e-02    0.994
60       2.291383e-02    1.402307e-02    0.994333
70       2.226722e-02    1.341250e-02    0.994667
80       2.183039e-02    1.292936e-02    0.994667
90       2.124771e-02    1.254406e-02    0.995
100      2.113763e-02    1.209475e-02    0.995333
110      2.038413e-02    1.170862e-02    0.995667
120      2.000775e-02    1.147197e-02    0.996
130      1.954538e-02    1.145159e-02    0.995667
140      1.908177e-02    1.134076e-02    0.996
150      1.862821e-02    1.102254e-02    0.995667
160      1.828849e-02    1.085362e-02    0.994667
170      1.802380e-02    1.069894e-02    0.995
180      1.771149e-02    1.049944e-02    0.995333
190      1.750447e-02    1.056114e-02    0.995
200      1.72511

KeyboardInterrupt: 

In [13]:
prediction = model(tensor_nd_test['x'])
print(prediction)

manual_check_numpy = prediction.detach().numpy()
manual_check = pd.DataFrame({
    "correct value": tensor_nd_test['L'].flatten(),
    "prediction": manual_check_numpy.flatten()})
manual_check.to_csv("manual_check.csv", index=False)

#data_pd_bench_sorted

prediction = model(tensor_nd_bench['x'])

submision_numpy = prediction.detach().numpy()

scale = data_pd_bench_sorted[8].to_numpy().reshape((-1,1))


if not labels:
    submision_numpy = np.multiply(submision_numpy , scale)#multiply by the length to turn it form dimensionles back to Area
    submision_numpy = np.multiply(submision_numpy , scale)
    
    submission_df = pd.DataFrame(submision_numpy)
    submission = pd.DataFrame({
        "id": range(prediction.shape[0]),
        "prediction": submision_numpy.flatten()})
    print(submission)
    submission.to_csv("predictions.csv", index=False)
    

else: 
    
    submission_df = pd.DataFrame(submision_numpy)
    submission = pd.DataFrame({
        "id": range(prediction.shape[0]),
        "prediction": submision_numpy.flatten()>0.5})
    print(submission)
submission.to_csv("predictions.csv", index=False)



tensor([[8.8714e-09],
        [1.0000e+00],
        [5.1253e-05],
        ...,
        [1.0000e+00],
        [1.0000e+00],
        [8.5223e-10]], grad_fn=<SigmoidBackward0>)
          id  prediction
0          0        True
1          1        True
2          2       False
3          3        True
4          4       False
...      ...         ...
44995  44995        True
44996  44996       False
44997  44997       False
44998  44998        True
44999  44999        True

[45000 rows x 2 columns]


In [ ]:
def MAPE_loss(pred, truth):
    return 100*torch.sum(torch.div(torch.abs(torch.sub(pred,truth)),truth))/pred.shape[0]


In [16]:
#Mape = 0.34 -> width = 140; epochs = 500, lr = 0.1e-3

width = 120
model = make_model_MAEBAPF(width)
epochs = 1000
print_every = epochs/100

lr = 1e-4
batch_size = 100
labels = False
#loss_fn = nn.L1Loss() # different loss function dont know which one to use

loss_fn = nn.MSELoss() # same loss function as in assignment 1

print(tensor_nd_train['x'].shape)
print(tensor_nd_train['y'].shape)
print(tensor_nd_train['L'].shape)
models_MAEBAPF, loss_dict_MAEBAPF = train_model(tensor_nd_train,tensor_nd_test,labels,model,loss_fn,epochs,lr,batch_size,print_every)

torch.Size([12000, 8])
torch.Size([12000, 1])
torch.Size([12000, 1])
Epoch    Train loss      Test loss       Test accuracy
10       6.502990e-04    1.406128e-03    0.0
20       2.461914e-04    5.455354e-04    0.0
30       1.284609e-04    2.578788e-04    0.0
40       8.692770e-05    1.456105e-04    0.0
50       6.565117e-05    1.035065e-04    0.0
60       5.016746e-05    8.097744e-05    0.0
70       3.961571e-05    6.432323e-05    0.0
80       3.335593e-05    5.521438e-05    0.0
90       2.806477e-05    4.706327e-05    0.0
100      2.400376e-05    4.011847e-05    0.0
110      2.126411e-05    3.574330e-05    0.0
120      1.913841e-05    3.210455e-05    0.0
130      1.748486e-05    2.953429e-05    0.0
140      1.621498e-05    2.716994e-05    0.0
150      1.516461e-05    2.516780e-05    0.0
160      1.425971e-05    2.341423e-05    0.0
170      1.337101e-05    2.207288e-05    0.0
180      1.250004e-05    2.081985e-05    0.0
190      1.157531e-05    1.973173e-05    0.0
200      1.118457e-05

KeyboardInterrupt: 

In [ ]:
plot_result_MAPE(tensor_nd_test, models_MAEBAPF)
plot_loss(loss_dict_MAEBAPF,False)

In [ ]:
def plot_loss(loss , labels:bool):
    epochs = np.arange(len(loss['train']))
    plt.plot(epochs, loss['train'], label="Train Loss")
    plt.plot(epochs, loss['test'], label="Test Loss")

    plt.xlabel("Epoch")
    if labels:
        plt.ylabel("BCE Loss")
    else:
        plt.ylabel("MSE Loss")
    plt.yscale("log")
    plt.xscale("log")
    plt.legend()

    plt.show()

In [ ]:

    
def plot_accuracies(test_acc, expected_value_random_guess = 0.5):
    plt.loglog(test_acc, label="test accuracy", marker=".", lw=2)
    plt.axhline(expected_value_random_guess, color='cyan', linestyle='--', label='Random Guess accuracy')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

In [ ]:
def plot_result_MSE(data_dict, model):
    with torch.no_grad():
        loss = loss_fn(y_pred, data_dict["y"])

        # print(loss.item())
        loss_amount = loss.item()
        print(f"Final MSE Loss: {loss_amount}")

        # plt.plot(y_pred.cpu().numpy(), data_dict["y_test"].cpu().numpy(), "o")
        plt.hist2d(
            data_dict["y"].cpu().numpy().flatten(),
            y_pred.cpu().numpy().flatten(),
            bins=50,
        )
        plt.xlabel("Predicted")
        plt.ylabel("Ground truth")
        plt.colorbar()
        plt.title(f"(Final MSE Loss: {loss_amount:.4f})")

        # plot line x = y
        x = [0, 12]
        plt.plot(x, x, "r")

        plt.show()
    return y_pred.cpu().numpy().flatten().T



In [ ]:
def plot_result_MAPE(data_dict, model):
    with torch.no_grad():
        y_pred = model(data_dict["x"])
        print(y_pred)
        print(data_dict["y"])
        loss = MAPE_loss(y_pred, data_dict["y"])
        print(loss)
        

        
        # print(loss.item())
        loss_amount = loss.item()
        print(f"Final MSE Loss: {loss_amount}")

        # plt.plot(y_pred.cpu().numpy(), data_dict["y_test"].cpu().numpy(), "o")
        plt.hist2d(
            data_dict["y"].cpu().numpy().flatten(),
            y_pred.cpu().numpy().flatten(),
            bins=50,
        )
        plt.xlabel("Predicted")
        plt.ylabel("Ground truth")
        plt.colorbar()
        plt.title(f"(Final MSE Loss: {loss_amount:.4f})")

        # plot line x = y
        x = [0, 12]
        plt.plot(x, x, "r")

        plt.show()
    return y_pred.cpu().numpy().flatten().T



In [ ]:
def predict_bench(data_dict, model):
    with torch.no_grad():
        y_pred = model(data_dict["x"])
    return y_pred.cpu().numpy().flatten().T


In [ ]:
def mape_singular(prediction, truth):
    return 100 * np.mean(np.abs((prediction - truth) / truth))

In [ ]:
#print(tensor_nd_test)
final_result = plot_result(tensor_nd_full, models_MAEBAPF)
final_prediction = predict_bench(tensor_nd_bench, models_MAEBAPF)
#print(final_result)
submission = pd.DataFrame({
    "id": range(len(final_result)),
    "prediction": final_result})
submission ['prediction'] = submission ['prediction'] # x8b
#print(submission['prediction'].to_numpy())
#print(tensor_nd_full['y'].cpu().numpy().flatten())
mape = mape_singular(submission['prediction'].to_numpy(), tensor_nd_full['y'].cpu().numpy().flatten())
print("Mape")
print(mape)

submission = pd.DataFrame({
    "id": range(len(final_prediction)),
    "prediction": final_prediction})
submission ['prediction'] = submission ['prediction'] * x8b

submission.to_csv("predictions.csv", index=False)

print(submission)